In [ ]:
# =====================================================================
# RoseSSL-CBAMNet  |  MULTI-SEED RUN (42, 123, 2024, 70)
# - ONE fixed 70/15/15 split (saved to disk) -> same 485-image test set for every seed
# - Only model randomness changes per seed (SSL pretrain + classifier init + shuffle)
# - Session-resumable: finished seeds / finished SSL stages are skipped after kernel reset
# - Reports correct/total per seed so accuracy = correct/485 is verifiable
# =====================================================================
import os, gc, json, random
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score,
    precision_score, recall_score, f1_score, roc_auc_score
)
from tensorflow.keras import layers, Model

# ------------------------- CONFIG -------------------------
SEEDS            = [42, 123, 2024, 70]
SPLIT_SEED       = 42
IMG_SIZE         = 224
BATCH_SIZE       = 32
SSL_EPOCHS       = 30
CLASSIFIER_EPOCHS = 30
TEMPERATURE      = 0.5
AUTOTUNE         = tf.data.AUTOTUNE
COMBINED_PATH    = "/kaggle/input/datasets/sabbir4724/rose-leaf"
OUT_DIR          = "/kaggle/working/multiseed"
os.makedirs(OUT_DIR, exist_ok=True)

print("TensorFlow:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))

# ------------------------- DATA + FIXED SPLIT -------------------------
class_names = sorted([d for d in os.listdir(COMBINED_PATH)
                      if os.path.isdir(os.path.join(COMBINED_PATH, d))])
class_to_idx = {c: i for i, c in enumerate(class_names)}
NUM_CLASSES = len(class_names)

image_paths, labels = [], []
for cls in class_names:
    folder = os.path.join(COMBINED_PATH, cls)
    for img in sorted(os.listdir(folder)):
        p = os.path.join(folder, img)
        if os.path.isfile(p):
            image_paths.append(p)
            labels.append(class_to_idx[cls])
image_paths = np.array(image_paths)
labels = np.array(labels)
print("Classes:", NUM_CLASSES, class_names)
print("Total images:", len(image_paths))

SPLIT_FILE = os.path.join(OUT_DIR, "split.json")
if os.path.exists(SPLIT_FILE):
    sp = json.load(open(SPLIT_FILE))
    train_paths, train_labels = np.array(sp["train_paths"]), np.array(sp["train_labels"])
    val_paths,   val_labels   = np.array(sp["val_paths"]),   np.array(sp["val_labels"])
    test_paths,  test_labels  = np.array(sp["test_paths"]),  np.array(sp["test_labels"])
    print("Loaded saved split.")
else:
    train_paths, temp_paths, train_labels, temp_labels = train_test_split(
        image_paths, labels, test_size=0.30, random_state=SPLIT_SEED, stratify=labels)
    val_paths, test_paths, val_labels, test_labels = train_test_split(
        temp_paths, temp_labels, test_size=0.50, random_state=SPLIT_SEED, stratify=temp_labels)
    json.dump({
        "train_paths": train_paths.tolist(), "train_labels": train_labels.tolist(),
        "val_paths": val_paths.tolist(),     "val_labels": val_labels.tolist(),
        "test_paths": test_paths.tolist(),   "test_labels": test_labels.tolist(),
    }, open(SPLIT_FILE, "w"))
    print("Created + saved split.")

print(f"Train={len(train_paths)}  Val={len(val_paths)}  Test={len(test_paths)}")
assert len(set(train_paths) & set(test_paths)) == 0
assert len(set(train_paths) & set(val_paths)) == 0
assert len(set(val_paths) & set(test_paths)) == 0
N_TEST = len(test_paths)


def load_image(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img.set_shape([None, None, 3])
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
    img = tf.cast(img, tf.float32)
    return img, label


def make_ds(paths, lbls, shuffle_seed=None):
    ds = tf.data.Dataset.from_tensor_slices((paths, lbls)).map(load_image, num_parallel_calls=AUTOTUNE)
    if shuffle_seed is not None:
        ds = ds.shuffle(len(paths), seed=shuffle_seed, reshuffle_each_iteration=True)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)


# ------------------------- MODEL PARTS -------------------------
class ChannelMean(layers.Layer):
    def call(self, x):
        return tf.reduce_mean(x, axis=-1, keepdims=True)


class ChannelMax(layers.Layer):
    def call(self, x):
        return tf.reduce_max(x, axis=-1, keepdims=True)


def cbam_block(input_feature, ratio=8):
    channel = input_feature.shape[-1]
    avg_pool = layers.Reshape((1, 1, channel))(layers.GlobalAveragePooling2D()(input_feature))
    max_pool = layers.Reshape((1, 1, channel))(layers.GlobalMaxPooling2D()(input_feature))
    d1 = layers.Dense(channel // ratio, activation="relu")
    d2 = layers.Dense(channel)
    ch_att = layers.Activation("sigmoid")(layers.Add()([d2(d1(avg_pool)), d2(d1(max_pool))]))
    ch_ref = layers.Multiply()([input_feature, ch_att])
    sp = layers.Concatenate(axis=-1)([ChannelMean()(ch_ref), ChannelMax()(ch_ref)])
    sp_att = layers.Conv2D(1, 7, padding="same", activation="sigmoid")(sp)
    return layers.Multiply()([ch_ref, sp_att])


def nt_xent_loss(zis, zjs, temperature=0.5):
    zis = tf.math.l2_normalize(zis, axis=1)
    zjs = tf.math.l2_normalize(zjs, axis=1)
    reps = tf.concat([zis, zjs], axis=0)
    sim = tf.matmul(reps, reps, transpose_b=True)
    bs = tf.shape(zis)[0]
    sim = sim - tf.eye(2 * bs) * 1e9
    positives = tf.concat([tf.range(bs, 2 * bs), tf.range(0, bs)], axis=0)
    loss = tf.keras.losses.sparse_categorical_crossentropy(positives, sim / temperature, from_logits=True)
    return tf.reduce_mean(loss)


def build_augmenter():
    return tf.keras.Sequential([
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.2),
        layers.RandomZoom(0.2),
        layers.RandomContrast(0.2),
        layers.RandomTranslation(0.1, 0.1),
    ])


# ------------------------- ONE SEED -------------------------
def run_seed(seed):
    sdir = os.path.join(OUT_DIR, f"seed_{seed}")
    os.makedirs(sdir, exist_ok=True)
    result_file = os.path.join(sdir, "result.json")
    if os.path.exists(result_file):
        print(f"[seed {seed}] already finished -> skipping")
        return json.load(open(result_file))

    print("\n" + "=" * 70)
    print(f"SEED {seed}")
    print("=" * 70)

    tf.keras.backend.clear_session()
    gc.collect()
    random.seed(seed)
    np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)   # python + numpy + tf

    train_ds = make_ds(train_paths, train_labels, shuffle_seed=seed)
    val_ds   = make_ds(val_paths, val_labels)
    test_ds  = make_ds(test_paths, test_labels)

    # ---------- Stage 1: SimCLR pretraining (resumable) ----------
    enc_path = os.path.join(sdir, "ssl_encoder.keras")
    loss_path = os.path.join(sdir, "ssl_losses.json")
    if os.path.exists(enc_path) and os.path.exists(loss_path):
        print("SSL encoder found -> skipping pretraining.")
        ssl_losses = json.load(open(loss_path))
    else:
        augmenter = build_augmenter()
        base_encoder = tf.keras.applications.MobileNetV2(
            include_top=False, weights="imagenet", input_shape=(IMG_SIZE, IMG_SIZE, 3))
        base_encoder.trainable = True

        inp = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
        x = tf.keras.applications.mobilenet_v2.preprocess_input(inp)
        x = base_encoder(x, training=True)
        x = layers.GlobalAveragePooling2D()(x)
        x = layers.Dense(512, activation="relu")(x)
        proj = layers.Dense(128)(x)
        simclr_model = Model(inp, proj)
        opt = tf.keras.optimizers.Adam(1e-4)

        @tf.function
        def train_step(images):
            a1 = augmenter(images, training=True)
            a2 = augmenter(images, training=True)
            with tf.GradientTape() as tape:
                z1 = simclr_model(a1, training=True)
                z2 = simclr_model(a2, training=True)
                loss = nt_xent_loss(z1, z2, TEMPERATURE)
            grads = tape.gradient(loss, simclr_model.trainable_variables)
            opt.apply_gradients(zip(grads, simclr_model.trainable_variables))
            return loss

        ssl_losses = []
        for ep in range(SSL_EPOCHS):
            bl = [train_step(imgs).numpy() for imgs, _ in train_ds]
            ssl_losses.append(float(np.mean(bl)))
            print(f"[seed {seed}] SSL epoch {ep + 1}/{SSL_EPOCHS}  loss={ssl_losses[-1]:.4f}")

        base_encoder.save(enc_path)
        json.dump(ssl_losses, open(loss_path, "w"))
        del simclr_model, base_encoder, opt
        gc.collect()

    plt.figure(figsize=(8, 5))
    plt.plot(ssl_losses)
    plt.title(f"SimCLR NT-Xent Loss (seed {seed})")
    plt.xlabel("Epoch"); plt.ylabel("Loss")
    plt.tight_layout()
    plt.savefig(os.path.join(sdir, "simclr_loss.png"), dpi=600)
    plt.close()

    # ---------- Stage 2: CBAM classifier ----------
    feature_extractor = tf.keras.models.load_model(enc_path)

    inp = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = tf.keras.applications.mobilenet_v2.preprocess_input(inp)
    x = feature_extractor(x, training=False)
    x = cbam_block(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(512, activation="relu", name="feature_dense")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)
    out = layers.Dense(NUM_CLASSES, activation="softmax")(x)
    model = Model(inp, out)

    model.compile(optimizer=tf.keras.optimizers.Adam(1e-4),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])

    best_w = os.path.join(sdir, "best_model.weights.h5")
    cbs = [
        tf.keras.callbacks.ModelCheckpoint(best_w, monitor="val_loss", save_best_only=True,
                                           save_weights_only=True, mode="min", verbose=0),
        tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=8, mode="min",
                                         restore_best_weights=True, verbose=1),
        tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.2, patience=3,
                                             mode="min", verbose=1),
    ]
    history = model.fit(train_ds, validation_data=val_ds, epochs=CLASSIFIER_EPOCHS, callbacks=cbs)
    model.load_weights(best_w)
    hist = history.history
    best_epoch = int(np.argmin(hist["val_loss"])) + 1

    for key, title in [("accuracy", "Accuracy"), ("loss", "Loss")]:
        plt.figure(figsize=(8, 5))
        plt.plot(hist[key], label="Train")
        plt.plot(hist["val_" + key], label="Validation")
        plt.title(f"{title} Curve (seed {seed})")
        plt.xlabel("Epoch"); plt.ylabel(title); plt.legend()
        plt.tight_layout()
        plt.savefig(os.path.join(sdir, f"{key}_curve.png"), dpi=600)
        plt.close()

    # ---------- Test evaluation (single pass, explicit counts) ----------
    y_prob = model.predict(test_ds, verbose=0)
    y_pred = np.argmax(y_prob, axis=1)
    y_true = np.concatenate([lb.numpy() for _, lb in test_ds])
    assert len(y_true) == N_TEST == len(y_pred)

    correct = int((y_true == y_pred).sum())
    total = int(len(y_true))
    acc = accuracy_score(y_true, y_pred)
    assert abs(acc - correct / total) < 1e-12

    res = {
        "seed": seed,
        "n_train": int(len(train_paths)), "n_val": int(len(val_paths)), "n_test": total,
        "correct": correct, "total": total,
        "accuracy": float(acc),
        "precision_w": float(precision_score(y_true, y_pred, average="weighted", zero_division=0)),
        "recall_w": float(recall_score(y_true, y_pred, average="weighted", zero_division=0)),
        "f1_w": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
        "f1_macro": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "auc_macro_ovr": float(roc_auc_score(y_true, y_prob, multi_class="ovr", average="macro")),
        "best_epoch": best_epoch,
        "epochs_run": len(hist["loss"]),
    }
    print(f"\n[seed {seed}] TEST: {correct}/{total} correct -> acc={acc:.4f} "
          f"| F1(w)={res['f1_w']:.4f} | F1(macro)={res['f1_macro']:.4f} | best_epoch={best_epoch}")

    rep = classification_report(y_true, y_pred, target_names=class_names, digits=4)
    print(rep)
    open(os.path.join(sdir, "test_classification_report.txt"), "w").write(rep)
    np.savez(os.path.join(sdir, "test_predictions.npz"), y_true=y_true, y_pred=y_pred, y_prob=y_prob)

    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(9, 7))
    plt.imshow(cm, cmap="Blues"); plt.colorbar()
    plt.xticks(range(NUM_CLASSES), class_names, rotation=90)
    plt.yticks(range(NUM_CLASSES), class_names)
    for i in range(NUM_CLASSES):
        for j in range(NUM_CLASSES):
            plt.text(j, i, cm[i, j], ha="center", va="center",
                     color="white" if cm[i, j] > cm.max() / 2 else "black")
    plt.xlabel("Predicted"); plt.ylabel("True")
    plt.title(f"Confusion Matrix - Test (seed {seed})")
    plt.tight_layout()
    plt.savefig(os.path.join(sdir, "test_confusion_matrix.png"), dpi=600)
    plt.close()

    json.dump(res, open(result_file, "w"), indent=2)   # written LAST = seed is "done"
    del model, feature_extractor
    tf.keras.backend.clear_session()
    gc.collect()
    return res


# ------------------------- RUN ALL SEEDS -------------------------
all_results = [run_seed(s) for s in SEEDS]

# ------------------------- AGGREGATE -------------------------
df = pd.DataFrame(all_results)
cols = ["seed", "n_test", "correct", "total", "accuracy", "precision_w", "recall_w",
        "f1_w", "f1_macro", "auc_macro_ovr", "best_epoch", "epochs_run"]
df = df[cols]
df.to_csv(os.path.join(OUT_DIR, "per_seed_results.csv"), index=False)

metric_cols = ["accuracy", "precision_w", "recall_w", "f1_w", "f1_macro", "auc_macro_ovr"]
summary = pd.DataFrame({
    "Metric": metric_cols,
    "Mean": [df[m].mean() for m in metric_cols],
    "Std (ddof=1)": [df[m].std(ddof=1) for m in metric_cols],
    "Min": [df[m].min() for m in metric_cols],
    "Max": [df[m].max() for m in metric_cols],
})
summary["Mean +- Std (%)"] = [
    f"{100 * r.Mean:.2f} +- {100 * r['Std (ddof=1)']:.2f}" for _, r in summary.iterrows()
]
summary.to_csv(os.path.join(OUT_DIR, "multiseed_summary.csv"), index=False)

print("\n" + "=" * 70)
print(f"PER-SEED RESULTS  (fixed test set = {N_TEST} images; acc = correct/{N_TEST})")
print("=" * 70)
print(df.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
print("\nSUMMARY ACROSS SEEDS")
print(summary.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

# per-seed accuracy plot
plt.figure(figsize=(7, 5))
plt.bar([str(s) for s in df["seed"]], df["accuracy"] * 100)
plt.axhline(df["accuracy"].mean() * 100, linestyle="--", label=f"Mean = {df['accuracy'].mean() * 100:.2f}%")
lo = max(0, df["accuracy"].min() * 100 - 2)
plt.ylim(lo, 100)
plt.xlabel("Seed"); plt.ylabel("Test Accuracy (%)")
plt.title(f"RoseSSL-CBAMNet - per-seed test accuracy (n={N_TEST})")
plt.legend(); plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "per_seed_accuracy.png"), dpi=600)
plt.show()

print(f"\nAll outputs saved in: {OUT_DIR}")